In [3]:
import pandas as pd
import numpy as np


In [ ]:
gz_path = r"D:\projects\street-level-crime-pipeline\data\raw\eurostat_files\estat_crim_off_cat.tsv"

df = pd.read_csv(gz_path, sep='\t')
cols = ["freq", "iccs", "unit", "geo"]
df[cols] = df['freq,iccs,unit,geo'].str.split(pat=",", expand=True)
cols.extend(df.filter(regex="[0-9]{4}").columns)
cols.remove('freq')
df = df[cols]
df.columns = [c.strip() for c in df.columns]
df = df.replace(": *", np.nan, regex=True)
df['unit'] = df['unit'].apply(lambda x: 'Absolute numbers' if x == 'NR' else 'Per 100000 inhabitants' if x == 'P_HTHAB' else "Unknown")
df = pd.melt(df, id_vars=['iccs', 'unit', 'geo'], value_vars=df.filter(regex="[0-9]{4}").columns, var_name='year', value_name='crime_count')

In [56]:
dfs = {}
for u in df['unit'].unique():
    dfs[u] = df[df['unit'] == u].copy()
    #print(df[df.isna()])

In [11]:
df.columns

Index(['iccs', 'unit', 'geo', 'Year', 'crime_count'], dtype='str')

In [6]:
df_codes = pd.read_csv(r"D:\projects\street-level-crime-pipeline\config\eu_crime\iccs_crime_codes.tsv", sep='\t')
dfc = df_codes[['Code', 'Label']].set_index('Code')
lookup_dict = dfc['Label'].to_dict()


In [101]:
lookup_dict

{'0101': 'Intentional homicide',
 '0102': 'Attempted intentional homicide',
 '0103': 'Non-intentional homicide',
 '01031': 'Non-negligent manslaughter',
 '01032': 'Negligent manslaughter',
 '010321': 'Vehicular homicide',
 '010322': 'Non-vehicular homicide',
 '0104': 'Assisting or instigating suicide',
 '01041': 'Assisting suicide',
 '01049': 'Other acts of assisting or instigating suicide',
 '0105': 'Euthanasia',
 '0106': 'Illegal feticide',
 '0107': 'Unlawful killing associated with armed conflict',
 'Other acts leading to death or intending to cause death': nan,
 '0201': 'Assaults and threats',
 '02011': 'Assault',
 '020111': 'Serious assault',
 '020112': 'Minor assault',
 '02012': 'Threat',
 '020121': 'Serious threat',
 '020122': 'Minor threat',
 '02019': 'Other assaults or threats',
 '0202': 'Acts against liberty',
 '02021': 'Abduction of a minor',
 '020211': 'Parental abduction',
 '020212': 'Abduction by another family member',
 '020213': 'Abduction by a legal guardian',
 '020219

In [7]:
x = 'ICCS0101'
#df_codes.loc[df_codes['Code'] == x.replace('ICCS', ''), 'Code'].values[0]


df['iccs'] = df['iccs'].apply(lambda x: lookup_dict[x.replace('ICCS', '')])

In [8]:
df

,iccs,unit,geo,Year,Count
0,Intentional homicide,Absolute numbers,AL,2008,88
1,Intentional homicide,Absolute numbers,AT,2008,58
2,Intentional homicide,Absolute numbers,BA,2008,66
3,Intentional homicide,Absolute numbers,BE,2008,204
4,Intentional homicide,Absolute numbers,BG,2008,172
...,...,...,...,...,...
30153,Acts that result in the depletion or degradati...,Per 100000 inhabitants,RO,2024,NaN
30154,Acts that result in the depletion or degradati...,Per 100000 inhabitants,RS,2024,NaN
30155,Acts that result in the depletion or degradati...,Per 100000 inhabitants,SE,2024,NaN
30156,Acts that result in the depletion or degradati...,Per 100000 inhabitants,SI,2024,NaN
